# 🤖 Retrieval-Augmented Generation (RAG)
## with Qwen LLM, FAISS Vector Store & Hugging Face Embeddings

---

This notebook demonstrates how to build a **complete end-to-end RAG application** from scratch.

Instead of relying on an LLM's training data alone, RAG allows the model to **retrieve relevant information from your own document** before generating an answer. This means you can ask questions about *any document you upload* — even if the LLM has never seen it before.

**What you will build:**
- Load your own `.txt` document from your local file system
- Split it into chunks and generate vector embeddings
- Store embeddings in a FAISS vector database
- Ask questions and get answers grounded in YOUR document
- Run an interactive multi-turn chat powered by Qwen via Groq

> 💡 **No GPU required.** This notebook runs entirely on your local CPU.

## 📚 Learning Objectives

After completing this notebook, you will understand:

| Concept | What You Will Learn |
|---|---|
| **What is RAG?** | How retrieval augments LLM generation with external knowledge |
| **Why RAG?** | Solving hallucination, knowledge cutoff, and private data challenges |
| **Document Ingestion** | Reading and preprocessing text documents |
| **Text Chunking** | Splitting documents into manageable, overlapping pieces |
| **Embeddings** | Converting text into numerical vectors for semantic search |
| **Vector Databases** | Storing and searching embeddings efficiently |
| **FAISS Similarity Search** | Finding the most relevant chunks for any query |
| **Context Retrieval** | Fetching the right context before answering |
| **Prompt Augmentation** | Building prompts that include retrieved context |
| **LLM Response Generation** | Generating grounded answers using Qwen via Groq |
| **Interactive RAG Chat** | Asking multiple questions in a chat interface |

## 🏗️ RAG Architecture Overview

RAG has two distinct phases:

### Phase 1 — Indexing (Run Once)
```
User Provides Text Document Path
          ↓
     Read Document
          ↓
     Text Cleaning
          ↓
    Text Chunking
          ↓
 Hugging Face Embeddings (MiniLM-L6-v2)
          ↓
      FAISS Vector DB  ← Ready for search!
```

### Phase 2 — Retrieval + Generation (Per Question)
```
       User Query
          ↓
 Query Embedding Generation
          ↓
 FAISS Similarity Search
          ↓
 Relevant Context Chunks
          ↓
   Augmented Prompt = System + Context + Question
          ↓
      Qwen LLM (via Groq)
          ↓
      Final Answer
```

### Stage-by-Stage Explanation

| Stage | What Happens |
|---|---|
| **Document Load** | User provides a local `.txt` file path |
| **Text Extraction** | Raw text is read and cleaned |
| **Text Chunking** | Document is split into overlapping chunks (e.g., 500 chars, 50 overlap) |
| **Embedding Generation** | Each chunk is converted into a 384-dimensional vector using MiniLM |
| **FAISS Indexing** | All vectors are stored in a FAISS index for fast similarity search |
| **Query Embedding** | The user's question is also converted into a vector |
| **Similarity Search** | FAISS finds the top-k chunks most similar to the query vector |
| **Context Retrieval** | Those chunks are assembled into a context string |
| **Prompt Augmentation** | Context + question are combined into a structured prompt |
| **LLM Generation** | Qwen reads the prompt and generates a grounded answer |

## 🆚 RAG vs. Normal LLM — Why Does RAG Matter?

### Normal LLM (No Retrieval)
```
Question → LLM → Answer
```
❌ Can only answer from training data  
❌ Has a knowledge cutoff date  
❌ Cannot access your private documents  
❌ May hallucinate facts it doesn't know  

### RAG (Retrieval-Augmented Generation)
```
Question
   ↓
Retriever → Relevant Document Chunks
   ↓
Context + Question → LLM → Grounded Answer
```
✅ Answers from YOUR document  
✅ Always up-to-date (you control the data)  
✅ Works with private/proprietary information  
✅ Reduced hallucination — answers grounded in evidence  

> 🔑 **Key Insight:** RAG allows you to inject external, private, or current information into an LLM *at inference time* — without retraining the model.

---
## ⚙️ Step 1 — Install Required Libraries

Run this cell once to install all dependencies into your local Python environment.

| Package | Purpose |
|---|---|
| `faiss-cpu` | Fast vector similarity search (Facebook AI) |
| `sentence-transformers` | Hugging Face MiniLM embedding model |
| `langchain` | RAG pipeline orchestration |
| `langchain-community` | Community integrations (FAISS wrapper) |
| `langchain-huggingface` | Hugging Face embeddings integration |
| `langchain-groq` | Groq LLM integration for LangChain |
| `groq` | Groq Python SDK |

> 💡 **Tip for VS Code users:** Make sure you have selected the correct Python interpreter/virtual environment in VS Code before running this cell (bottom-left status bar → select interpreter).

This cell only needs to be run **once** per environment. You can comment it out after the first successful run.

In [1]:
# Install all required dependencies into your local Python environment
# Run this cell once; you can comment it out after the first successful install
import sys

%pip install faiss-cpu
%pip install sentence-transformers
%pip install langchain
%pip install langchain-community
%pip install langchain-huggingface
%pip install langchain-groq
%pip install groq

print("✅ All libraries installed successfully!")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
✅ All libraries installed successfully!



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## 📦 Step 2 — Import Libraries & Global Configuration

All imports and configuration values are centralised here so you can easily modify them.

**Key configuration parameters:**
- `MODEL_NAME` — The Qwen model served through Groq
- `CHUNK_SIZE` — Maximum characters per document chunk
- `CHUNK_OVERLAP` — Overlapping characters between adjacent chunks (prevents losing context at boundaries)
- `TOP_K` — Number of chunks to retrieve per query
- `TEMPERATURE` — LLM creativity (0 = deterministic, 1 = creative)
- `SHOW_RETRIEVED_CONTEXT` — Toggle to display retrieved chunks alongside answers

In [2]:
# ── Standard Library ──────────────────────────────────────────────────────────
import os
import re
import warnings
from getpass import getpass

# ── LangChain ─────────────────────────────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════════════════════════════
#  GLOBAL CONFIGURATION  —  Edit these values to customise RAG behaviour
# ══════════════════════════════════════════════════════════════════════════════

# LLM: Qwen model served via Groq
# NOTE: Groq model identifiers follow the format "provider/model-name".
# At the time of writing, Groq supports Qwen models under this naming convention.
# If this identifier is rejected by the API, Groq's model list can be checked at:
# https://console.groq.com/docs/models
MODEL_NAME   = "qwen/qwen3.6-27b"       # Primary requested model (qwen3-8b on Groq)
TEMPERATURE  = 0.2                    # Low temperature for factual, grounded answers
MAX_TOKENS   = 1024                   # Maximum tokens in the generated answer

# Embedding model
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Chunking parameters
CHUNK_SIZE    = 500   # Characters per chunk
CHUNK_OVERLAP = 50    # Overlap between consecutive chunks

# Retrieval
TOP_K = 3             # Number of chunks to retrieve per query

# Educational toggle: show retrieved chunks alongside answers
SHOW_RETRIEVED_CONTEXT = True

# Supported file extensions
SUPPORTED_EXTENSIONS = [".txt", ".md"]

print("✅ Libraries imported and configuration loaded.")
print(f"   Model       : {MODEL_NAME}")
print(f"   Chunk Size  : {CHUNK_SIZE} characters")
print(f"   Chunk Overlap: {CHUNK_OVERLAP} characters")
print(f"   Top-K       : {TOP_K} chunks retrieved per query")
print(f"   Show Context: {SHOW_RETRIEVED_CONTEXT}")

c:\Users\Asus\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Asus\AppData\Local\Temp\ipykernel_28452\2770653531.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


✅ Libraries imported and configuration loaded.
   Model       : qwen/qwen3.6-27b
   Chunk Size  : 500 characters
   Chunk Overlap: 50 characters
   Top-K       : 3 chunks retrieved per query
   Show Context: True


---
## 📄 Step 3 — Load Your Text Document

Set the `FILE_PATH` variable below to the **absolute or relative path** of your `.txt` or `.md` file on your local machine.

**Good examples to try:**
- A company policy document
- A research paper (converted to .txt)
- A product manual
- Meeting notes
- Any article or report

**Examples of valid paths:**
```python
FILE_PATH = "C:/Users/YourName/Documents/my_document.txt"   # Windows absolute path
FILE_PATH = "/home/yourname/documents/my_document.txt"       # Linux/macOS absolute path
FILE_PATH = "my_document.txt"                                 # Relative to notebook directory
FILE_PATH = "../data/my_document.txt"                         # Relative parent directory
```

In [3]:
# ── Set the path to your local document ───────────────────────────────────────
# Edit this variable to point to your .txt or .md file
FILE_PATH = "source.txt"   # <-- CHANGE THIS to your file path

# ── Validate file path ────────────────────────────────────────────────────────
if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f"❌ File not found: '{FILE_PATH}'\n"
        f"   Please update FILE_PATH to point to your document."
    )

file_name = os.path.basename(FILE_PATH)
file_ext  = os.path.splitext(file_name)[1].lower()

if file_ext not in SUPPORTED_EXTENSIONS:
    raise ValueError(
        f"❌ Unsupported file type: '{file_ext}'.\n"
        f"   Please provide a file of type: {SUPPORTED_EXTENSIONS}"
    )

print(f"✅ File found: {FILE_PATH}")
print(f"   File name : {file_name}")
print(f"   File size : {os.path.getsize(FILE_PATH):,} bytes")

✅ File found: source.txt
   File name : source.txt
   File size : 11,702 bytes


---
## 🔍 Step 4 — Read & Inspect the Document

We read the file from disk, decode it as UTF-8 text, and display basic statistics.

Only a short preview is shown to avoid flooding the notebook with large documents.

In [4]:
# ── Read the file from disk ───────────────────────────────────────────────────
try:
    with open(FILE_PATH, "r", encoding="utf-8") as f:
        raw_text = f.read()
except UnicodeDecodeError:
    # Fall back to latin-1 if UTF-8 fails
    with open(FILE_PATH, "r", encoding="latin-1") as f:
        raw_text = f.read()
    print("⚠️  UTF-8 decoding failed — used latin-1 fallback.")

# ── Basic cleaning ─────────────────────────────────────────────────────────────
# Remove excessive blank lines while preserving paragraph structure
cleaned_text = re.sub(r'\n{3,}', '\n\n', raw_text).strip()

if not cleaned_text:
    raise ValueError("❌ The document appears to be empty. Please provide a document with content.")

# ── Statistics ─────────────────────────────────────────────────────────────────
char_count  = len(cleaned_text)
word_count  = len(cleaned_text.split())
line_count  = len(cleaned_text.splitlines())

print("=" * 52)
print("📄 DOCUMENT STATISTICS")
print("=" * 52)
print(f"  File name  : {file_name}")
print(f"  Characters : {char_count:,}")
print(f"  Words      : {word_count:,}")
print(f"  Lines      : {line_count:,}")
print("=" * 52)

# ── Preview ────────────────────────────────────────────────────────────────────
preview_chars = 500
print("\n📖 Document Preview (first 500 characters):")
print("-" * 52)
print(cleaned_text[:preview_chars])
if char_count > preview_chars:
    print(f"\n... [{char_count - preview_chars:,} more characters not shown]")

📄 DOCUMENT STATISTICS
  File name  : source.txt
  Characters : 11,557
  Words      : 1,708
  Lines      : 118

📖 Document Preview (first 500 characters):
----------------------------------------------------
India: A Comprehensive Overview

I. Introduction

India, officially known as the Republic of India (Bharat), is a nation of monumental scale, staggering diversity, and profound historical significance. Located in South Asia, it is the seventh-largest country by land area and the most populous democracy in the world. With a population exceeding 1.4 billion people, India represents a microcosm of humanity, encompassing a vast array of ethnicities, languages, religions, and socio-economic strata. I

... [11,057 more characters not shown]


---
## ✂️ Step 5 — Text Chunking

### Why do we split the document into chunks?

LLMs have a **context window limit** — they cannot process an entire large document at once. More importantly, for RAG to work effectively, we need to retrieve only the *most relevant* piece of the document for each question. Smaller, focused chunks make retrieval more precise.

### Why overlap?

Imagine a sentence that spans the boundary between two chunks. Without overlap, that sentence would be split in half, losing its meaning. **Overlap ensures boundary information is captured in both adjacent chunks.**

### `RecursiveCharacterTextSplitter`

This splitter tries to split on natural boundaries first (paragraphs → sentences → words → characters), producing more semantically coherent chunks than a simple fixed-size split.

```
┌─────────────────────────────────────────────┐
│              Full Document                  │
└────────────────┬────────────────────────────┘
                 ↓  RecursiveCharacterTextSplitter
   ┌──────┐  ┌──────┐  ┌──────┐  ┌──────┐
   │Chunk1│  │Chunk2│  │Chunk3│  │Chunk4│  ...
   └──────┘  └──────┘  └──────┘  └──────┘
         ↑──50 char overlap──↑
```

In [5]:
# ── Create the text splitter ───────────────────────────────────────────────────
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size    = CHUNK_SIZE,
    chunk_overlap = CHUNK_OVERLAP,
    length_function = len,
    separators = ["\n\n", "\n", ". ", " ", ""]  # Try paragraph → sentence → word splits
)

# ── Split document ─────────────────────────────────────────────────────────────
chunks = text_splitter.split_text(cleaned_text)

if not chunks:
    raise ValueError("❌ No chunks were created. The document may be too short or empty.")

print(f"✅ Total chunks created: {len(chunks)}")
print(f"   Chunk size    : {CHUNK_SIZE} characters")
print(f"   Chunk overlap : {CHUNK_OVERLAP} characters")

# ── Display sample chunks ──────────────────────────────────────────────────────
sample_count = min(3, len(chunks))
print(f"\n📋 Sample Chunks (showing {sample_count} of {len(chunks)}):")
print("=" * 60)
for i in range(sample_count):
    print(f"\n--- Chunk {i + 1} (length: {len(chunks[i])} chars) ---")
    print(chunks[i])
print("=" * 60)

✅ Total chunks created: 33
   Chunk size    : 500 characters
   Chunk overlap : 50 characters

📋 Sample Chunks (showing 3 of 33):

--- Chunk 1 (length: 48 chars) ---
India: A Comprehensive Overview

I. Introduction

--- Chunk 2 (length: 447 chars) ---
India, officially known as the Republic of India (Bharat), is a nation of monumental scale, staggering diversity, and profound historical significance. Located in South Asia, it is the seventh-largest country by land area and the most populous democracy in the world. With a population exceeding 1.4 billion people, India represents a microcosm of humanity, encompassing a vast array of ethnicities, languages, religions, and socio-economic strata

--- Chunk 3 (length: 256 chars) ---
. It is bounded by the Indian Ocean on the south, the Arabian Sea on the southwest, and the Bay of Bengal on the southeast, while sharing land borders with Pakistan to the west; China, Nepal, and Bhutan to the north; and Bangladesh and Myanmar to the east.


---
## 🔢 Step 6 — Generate Embeddings

### What is an embedding?

An **embedding** is a numerical representation (vector) of text. The key property is that *semantically similar text produces similar vectors*.

```
"The dog barked loudly"   →  [0.12, -0.45, 0.88, ...]  (384 numbers)
"A dog was making noise"  →  [0.11, -0.43, 0.85, ...]  ← Very similar!
"Stock market rose today" →  [-0.71, 0.22, -0.39, ...] ← Very different
```

This allows us to find relevant chunks using **semantic similarity** rather than exact keyword matching.

### Why `all-MiniLM-L6-v2`?

- 🚀 Fast and lightweight (perfect for CPU)
- 📐 Produces 384-dimensional embeddings
- 🎯 Excellent semantic similarity quality for its size
- 🆓 Freely available on Hugging Face

```
Text Chunk
    ↓
all-MiniLM-L6-v2
    ↓
[v₁, v₂, v₃, ..., v₃₈₄]  ← 384-dimensional vector
```

> ⏳ First run downloads the model (~90 MB) to your local Hugging Face cache. Subsequent runs use the cache.

In [6]:
print(f"🔄 Loading embedding model: {EMBEDDING_MODEL}")
print("   (First run downloads ~90 MB — please wait)\n")

try:
    embedding_model = HuggingFaceEmbeddings(
        model_name = EMBEDDING_MODEL,
        model_kwargs = {"device": "cpu"},   # Use CPU — no GPU required
        encode_kwargs = {"normalize_embeddings": True}  # L2-normalise for cosine similarity
    )
except Exception as e:
    raise RuntimeError(f"❌ Failed to load embedding model: {e}")

# ── Demonstrate a single embedding ────────────────────────────────────────────
demo_text       = "This is a demonstration of how text becomes a vector."
demo_embedding  = embedding_model.embed_query(demo_text)
embedding_dim   = len(demo_embedding)

print("✅ Embedding model loaded successfully!")
print(f"\n📐 Embedding Demonstration:")
print(f"   Input text      : \"{demo_text}\"")
print(f"   Embedding dim   : {embedding_dim}")
print(f"   First 8 values  : {[round(v, 4) for v in demo_embedding[:8]]}")
print(f"   ... [{embedding_dim - 8} more dimensions not shown]")

🔄 Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
   (First run downloads ~90 MB — please wait)



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2630.28it/s]


✅ Embedding model loaded successfully!

📐 Embedding Demonstration:
   Input text      : "This is a demonstration of how text becomes a vector."
   Embedding dim   : 384
   First 8 values  : [-0.0254, 0.0571, -0.0163, 0.0296, 0.0117, -0.0054, 0.0162, -0.009]
   ... [376 more dimensions not shown]


---
## 🗄️ Step 7 — Create FAISS Vector Store

### What is FAISS?

**FAISS** (Facebook AI Similarity Search) is a highly optimised library for fast nearest-neighbour search over dense vectors. It is the vector database we use to store all our chunk embeddings.

```
Document Chunks  →  Embedding Model  →  Vectors  →  FAISS Index
[chunk_1]                                [v_1]     ┌──────────┐
[chunk_2]        all-MiniLM-L6-v2       [v_2]  →  │  FAISS   │
[chunk_3]                                [v_3]     │  Index   │
  ...                                    ...       └──────────┘
```

### How similarity search works

When you ask a question, it is also converted to a vector. FAISS then finds the stored chunk vectors that are **closest** (most similar) to the question vector using cosine or L2 distance.

```
Query vector  ──→  FAISS  ──→  Top-K most similar chunk vectors  ──→  Text chunks
```

> ⏳ FAISS embeds all chunks now — this may take a moment for large documents.

In [7]:
print(f"🔄 Building FAISS index over {len(chunks)} chunks ...")

try:
    vector_store = FAISS.from_texts(
        texts      = chunks,
        embedding  = embedding_model
    )
except Exception as e:
    raise RuntimeError(f"❌ Failed to create FAISS vector store: {e}")

print("✅ FAISS vector store created successfully!")
print()
print("=" * 44)
print("🗄️  VECTOR STORE SUMMARY")
print("=" * 44)
print(f"  Vector store      : FAISS")
print(f"  Indexed chunks    : {len(chunks)}")
print(f"  Embedding model   : all-MiniLM-L6-v2")
print(f"  Embedding dim     : {embedding_dim}")
print(f"  Retrieval top-k   : {TOP_K}")
print("=" * 44)
print()
print("ℹ️  The FAISS index is held in memory.")
print("   All chunks are searchable via semantic similarity.")

🔄 Building FAISS index over 33 chunks ...
✅ FAISS vector store created successfully!

🗄️  VECTOR STORE SUMMARY
  Vector store      : FAISS
  Indexed chunks    : 33
  Embedding model   : all-MiniLM-L6-v2
  Embedding dim     : 384
  Retrieval top-k   : 3

ℹ️  The FAISS index is held in memory.
   All chunks are searchable via semantic similarity.


### 💾 Optional — Save & Load FAISS Index (Persistence)

Unlike in Colab where the runtime resets, your local environment can **persist** the FAISS index to disk. This lets you skip re-indexing on subsequent runs.

```python
# Save the index to disk
vector_store.save_local("faiss_index")

# Load the index from disk in a future session
vector_store = FAISS.load_local("faiss_index", embedding_model, allow_dangerous_deserialization=True)
```

---
## 🔑 Step 8 — Configure Groq API Key

Groq provides extremely fast LLM inference. We use the Qwen model through Groq's API.

**Get your free API key at:** [https://console.groq.com](https://console.groq.com)

**Options to provide your API key locally:**

1. **Prompted input (below)** — `getpass` will prompt you securely at runtime. The key is not stored.
2. **Environment variable (recommended)** — Set `GROQ_API_KEY` in your shell before launching VS Code/Jupyter:
   ```bash
   # Windows (PowerShell)
   $env:GROQ_API_KEY = "gsk_..."
   # macOS/Linux
   export GROQ_API_KEY="gsk_..."
   ```
3. **`.env` file** — Create a `.env` file in the notebook directory and load it with `python-dotenv`.

> ⚠️ **Security Notice:** Never hard-code API keys in notebooks. Never commit API keys to version control.

In [8]:
# Check if GROQ_API_KEY is already set as an environment variable
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

if not GROQ_API_KEY:
    # Securely prompt for the Groq API key — it will NOT be displayed or stored
    GROQ_API_KEY = getpass("🔑 Enter your Groq API Key: ")

if not GROQ_API_KEY or len(GROQ_API_KEY.strip()) < 10:
    raise ValueError("❌ API key appears invalid. Please provide a valid Groq API key.")

# Set as environment variable (good practice for SDK compatibility)
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("✅ Groq API key accepted (not displayed for security).")

✅ Groq API key accepted (not displayed for security).


---
## 🤖 Step 9 — Initialize Qwen LLM via Groq

We initialise the **Qwen** model through LangChain's Groq integration.

**Model note:**
The primary configured model is `qwen/qwen3-8b`. Groq's available Qwen models may vary over time. If the model identifier changes, update `MODEL_NAME` in the configuration cell above.

**Temperature = 0.2** keeps answers focused and factual — ideal for document Q&A where we want grounded, precise answers rather than creative responses.

In [9]:
print(f"🔄 Initializing LLM: {MODEL_NAME} via Groq ...")

try:
    llm = ChatGroq(
        model       = MODEL_NAME,
        temperature = TEMPERATURE,
        max_tokens  = MAX_TOKENS,
        api_key     = GROQ_API_KEY
    )
except Exception as e:
    raise RuntimeError(
        f"❌ Failed to initialize Groq LLM.\n"
        f"   Model attempted: {MODEL_NAME}\n"
        f"   Error: {e}\n\n"
        f"   ➡  Check available models at: https://console.groq.com/docs/models\n"
        f"   ➡  Update MODEL_NAME in the configuration cell if needed."
    )

print("✅ LLM initialized successfully!")
print(f"   Model       : {MODEL_NAME}")
print(f"   Provider    : Groq")
print(f"   Temperature : {TEMPERATURE}")
print(f"   Max tokens  : {MAX_TOKENS}")

🔄 Initializing LLM: qwen/qwen3.6-27b via Groq ...
✅ LLM initialized successfully!
   Model       : qwen/qwen3.6-27b
   Provider    : Groq
   Temperature : 0.2
   Max tokens  : 1024


---
## 🔎 Step 10 — Retrieval Function

### How retrieval works

The retrieval function is the **heart of RAG**. It:

1. Takes the user's question as input
2. Converts it to an embedding vector (same model as the chunks)
3. Searches FAISS for the `k` most similar chunk vectors
4. Returns the corresponding text chunks as context

```
User Question (text)
       ↓
all-MiniLM-L6-v2  →  Query Vector
       ↓
FAISS.similarity_search(query_vector, k=3)
       ↓
Top-3 Most Relevant Chunks
```

We demonstrate retrieval **independently** here so you can see what the LLM receives before it generates an answer.

In [10]:
def retrieve_context(query: str, k: int = TOP_K) -> tuple:
    """
    Retrieve the top-k most relevant chunks for a given query.

    Parameters
    ----------
    query : str   — The user's question
    k     : int   — Number of chunks to retrieve

    Returns
    -------
    context : str          — Retrieved chunks joined as a single context string
    docs    : list         — Raw LangChain Document objects (for inspection)
    """
    if not query.strip():
        raise ValueError("Query cannot be empty.")

    # Perform semantic similarity search in FAISS
    docs = vector_store.similarity_search(query, k=k)

    # Join retrieved chunks with clear separators
    context_parts = [f"[Chunk {i+1}]\n{doc.page_content}" for i, doc in enumerate(docs)]
    context = "\n\n".join(context_parts)

    return context, docs


# ── Demonstrate retrieval independently (educational) ─────────────────────────
print("📋 RETRIEVAL DEMONSTRATION")
print("=" * 60)
print("We will now perform a retrieval WITHOUT calling the LLM.")
print("This shows exactly what context the LLM will receive.\n")

demo_question = "What is the main topic of this document?"
print(f"Demo Question: {demo_question}\n")

demo_context, demo_docs = retrieve_context(demo_question)

print(f"✅ Retrieved {len(demo_docs)} chunk(s) from FAISS:")
print("-" * 60)
print(demo_context)
print("-" * 60)
print("\nℹ️  These chunks will be passed to the LLM as context.")

📋 RETRIEVAL DEMONSTRATION
We will now perform a retrieval WITHOUT calling the LLM.
This shows exactly what context the LLM will receive.

Demo Question: What is the main topic of this document?

✅ Retrieved 3 chunk(s) from FAISS:
------------------------------------------------------------
[Chunk 1]
India: A Comprehensive Overview

I. Introduction

[Chunk 2]
V. Political Structure

India is the world's largest democracy. It is a Sovereign, Socialist, Secular, Democratic Republic with a parliamentary system of government.

The Constitution: Adopted in 1950, it is the longest written constitution of any sovereign country. It guarantees fundamental rights (speech, equality, religion) to all citizens.

[Chunk 3]
II. Historical Tapestry

The history of India is a chronicle of the rise and fall of great empires, the birth of profound philosophies, and the convergence of diverse cultures.
------------------------------------------------------------

ℹ️  These chunks will be passed to the LLM 

---
## 📝 Step 11 — RAG Prompt Construction

### What is prompt augmentation?

The **augmented prompt** is the full message we send to the LLM. It has three parts:

```
┌─────────────────────────────────────┐
│  SYSTEM INSTRUCTION                 │  ← Tells the LLM how to behave
│  "Answer using ONLY the context..." │
├─────────────────────────────────────┤
│  RETRIEVED CONTEXT                  │  ← Evidence from YOUR document
│  [Chunk 1] ...                      │
│  [Chunk 2] ...                      │
│  [Chunk 3] ...                      │
├─────────────────────────────────────┤
│  USER QUESTION                      │  ← What the user asked
│  "What is the main purpose?"        │
└─────────────────────────────────────┘
```

**Why do we explicitly instruct the model to not hallucinate?**

Without the instruction, the LLM might combine retrieved context with its training data and invent plausible-sounding facts. The system instruction anchors the model to the retrieved context only.

In [11]:
# System instruction — defines LLM behaviour for this RAG application
SYSTEM_PROMPT = """You are a precise, helpful question-answering assistant.

Your task is to answer the user's question using ONLY the context provided below.

STRICT RULES:
1. Answer using ONLY the information present in the provided context.
2. Do NOT use your general training knowledge to supplement the answer.
3. Do NOT invent, assume, or extrapolate information not present in the context.
4. If the answer cannot be found in the context, respond exactly:
   "I could not find the answer in the uploaded document."
5. Keep your answer concise, clear, and directly relevant to the question.
6. If quoting from the context, do so accurately."""


def build_rag_prompt(context: str, question: str) -> list:
    """
    Build the list of LangChain messages for the augmented RAG prompt.

    Returns a list of [SystemMessage, HumanMessage] suitable for ChatGroq.
    """
    user_message = f"""Use the following context to answer the question.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:"""

    return [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_message)
    ]


# ── Show the augmented prompt structure ────────────────────────────────────────
print("📋 AUGMENTED PROMPT STRUCTURE")
print("=" * 60)
print("[SYSTEM MESSAGE]")
print(SYSTEM_PROMPT[:200] + " ...")
print()
print("[HUMAN MESSAGE — Template]")
print("Use the following context to answer the question.")
print()
print("CONTEXT:")
print("[Retrieved chunks from FAISS will appear here]")
print()
print("QUESTION:")
print("[User's question will appear here]")
print()
print("ANSWER:")
print("[LLM generates from this point]")
print("=" * 60)
print("✅ RAG prompt builder ready.")

📋 AUGMENTED PROMPT STRUCTURE
[SYSTEM MESSAGE]
You are a precise, helpful question-answering assistant.

Your task is to answer the user's question using ONLY the context provided below.

STRICT RULES:
1. Answer using ONLY the information present  ...

[HUMAN MESSAGE — Template]
Use the following context to answer the question.

CONTEXT:
[Retrieved chunks from FAISS will appear here]

QUESTION:
[User's question will appear here]

ANSWER:
[LLM generates from this point]
✅ RAG prompt builder ready.


---
## 🤖 Step 12 — RAG Question-Answer Function

This is the complete RAG pipeline assembled into a single reusable function:

```
ask_rag(question)
    │
    ├─ retrieve_context(question)     ← Step 1: Find relevant chunks
    ├─ build_rag_prompt(context, q)   ← Step 2: Build augmented prompt
    └─ llm.invoke(messages)           ← Step 3: Generate answer
```

In [12]:
def ask_rag(question: str, k: int = TOP_K, show_context: bool = SHOW_RETRIEVED_CONTEXT) -> str:
    """
    Full RAG pipeline: retrieve → augment → generate.

    Parameters
    ----------
    question     : str   — The user's question
    k            : int   — Number of chunks to retrieve
    show_context : bool  — If True, print retrieved chunks before the answer

    Returns
    -------
    answer : str  — The LLM's generated answer
    """
    if not question or not question.strip():
        return "⚠️  Please enter a non-empty question."

    try:
        # ── Step 1: Retrieve relevant context from FAISS ───────────────────────
        context, docs = retrieve_context(question, k=k)

        # ── Optional: Display retrieved context (educational) ──────────────────
        if show_context:
            print("📚 RETRIEVED CONTEXT")
            print("-" * 56)
            print(context)
            print("-" * 56)
            print()

        # ── Step 2: Build augmented prompt ─────────────────────────────────────
        messages = build_rag_prompt(context, question)

        # ── Step 3: Invoke LLM ─────────────────────────────────────────────────
        response = llm.invoke(messages)
        answer   = response.content.strip()

        return answer

    except Exception as e:
        return f"❌ Error generating answer: {e}"


print("✅ RAG function ready. The complete pipeline is:")
print()
print("  ask_rag(question)")
print("      │")
print("      ├─ 1. Embed question with MiniLM")
print("      ├─ 2. Search FAISS → top-k chunks")
print("      ├─ 3. Build augmented prompt")
print("      └─ 4. Qwen via Groq → answer")

✅ RAG function ready. The complete pipeline is:

  ask_rag(question)
      │
      ├─ 1. Embed question with MiniLM
      ├─ 2. Search FAISS → top-k chunks
      ├─ 3. Build augmented prompt
      └─ 4. Qwen via Groq → answer


---
## 🧪 Step 13 — Example Questions

We demonstrate three types of questions to illustrate RAG behaviour:

1. **Factual question** — Should be answered from document content
2. **Detail retrieval** — Tests multi-chunk retrieval
3. **Out-of-context question** — Should trigger the "not found" response, demonstrating hallucination prevention

> Observe how the **retrieved context** (when `SHOW_RETRIEVED_CONTEXT = True`) directly supports the generated answer.

In [13]:
# Example 1 — Direct factual question
print("=" * 60)
print("EXAMPLE 1 — Factual Question")
print("=" * 60)

q1 = "What is the main objective mentioned in the document?"
print(f"\nQuestion: {q1}\n")

a1 = ask_rag(q1)
print("🤖 Answer:")
print(a1)

EXAMPLE 1 — Factual Question

Question: What is the main objective mentioned in the document?

📚 RETRIEVED CONTEXT
--------------------------------------------------------
[Chunk 1]
India: A Comprehensive Overview

I. Introduction

[Chunk 2]
V. Political Structure

India is the world's largest democracy. It is a Sovereign, Socialist, Secular, Democratic Republic with a parliamentary system of government.

The Constitution: Adopted in 1950, it is the longest written constitution of any sovereign country. It guarantees fundamental rights (speech, equality, religion) to all citizens.

[Chunk 3]
Federal Structure: The country is a union of 28 States and 8 Union Territories. Power is distributed between the Central Government and the State Governments.

The Executive: The President is the ceremonial head of state. Real executive power resides with the Prime Minister and the Council of Ministers.

The Legislature: The Parliament is bicameral, consisting of the Lok Sabha (House of the People/

In [14]:
# Example 2 — Detail retrieval
print("=" * 60)
print("EXAMPLE 2 — Detail Retrieval")
print("=" * 60)

q2 = "What are the key responsibilities or activities described in the document?"
print(f"\nQuestion: {q2}\n")

a2 = ask_rag(q2)
print("🤖 Answer:")
print(a2)

EXAMPLE 2 — Detail Retrieval

Question: What are the key responsibilities or activities described in the document?

📚 RETRIEVED CONTEXT
--------------------------------------------------------
[Chunk 1]
India: A Comprehensive Overview

I. Introduction

[Chunk 2]
Federal Structure: The country is a union of 28 States and 8 Union Territories. Power is distributed between the Central Government and the State Governments.

The Executive: The President is the ceremonial head of state. Real executive power resides with the Prime Minister and the Council of Ministers.

The Legislature: The Parliament is bicameral, consisting of the Lok Sabha (House of the People/Lower House) and the Rajya Sabha (Council of States/Upper House).

[Chunk 3]
V. Political Structure

India is the world's largest democracy. It is a Sovereign, Socialist, Secular, Democratic Republic with a parliamentary system of government.

The Constitution: Adopted in 1950, it is the longest written constitution of any sovereign c

In [15]:
# Example 3 — Out-of-context question (hallucination prevention test)
print("=" * 60)
print("EXAMPLE 3 — Out-of-Context Question")
print("(Testing hallucination prevention)")
print("=" * 60)

q3 = "What is the current population of Japan?"
print(f"\nQuestion: {q3}")
print("(This is unlikely to be in your document!)\n")

a3 = ask_rag(q3)
print("🤖 Answer:")
print(a3)
print()
print("ℹ️  A well-behaved RAG system should respond that this")
print("   information is not available in the uploaded document,")
print("   rather than inventing an answer.")

EXAMPLE 3 — Out-of-Context Question
(Testing hallucination prevention)

Question: What is the current population of Japan?
(This is unlikely to be in your document!)

📚 RETRIEVED CONTEXT
--------------------------------------------------------
[Chunk 1]
Population and Urbanization: Managing a population of 1.4 billion requires massive resources. Rapid urbanization has put strain on infrastructure, leading to pollution and housing shortages in mega-cities like Delhi, Mumbai, and Bangalore.

Employment: Creating enough quality jobs for its "demographic dividend" (a massive young population) is a critical challenge.

[Chunk 2]
India, officially known as the Republic of India (Bharat), is a nation of monumental scale, staggering diversity, and profound historical significance. Located in South Asia, it is the seventh-largest country by land area and the most populous democracy in the world. With a population exceeding 1.4 billion people, India represents a microcosm of humanity, encompassi

---
## 💬 Step 14 — Interactive RAG Chat

You can now ask **unlimited questions** about your document in a chat loop.

**Instructions:**
- Type your question and press **Enter**
- Type `exit`, `quit`, or `bye` to end the chat
- The assistant can only answer from your document

```
You: [your question]
Assistant: [grounded answer from your document]
```

> 💡 Tip: Set `SHOW_RETRIEVED_CONTEXT = False` in the configuration cell (Step 2) to see cleaner chat output without the retrieved chunks displayed.

> 💡 **VS Code tip:** The `input()` prompt appears in the terminal/output panel below the cell. Click on it and type your question there.

In [16]:
print("=" * 56)
print("💬  RAG CHAT — Powered by Qwen via Groq")
print("=" * 56)
print(f"📄  Knowledge source : {file_name}")
print(f"🗄️  Vector store     : FAISS ({len(chunks)} chunks)")
print(f"🤖  Model           : {MODEL_NAME}")
print("=" * 56)
print("\nAsk questions about your document.")
print("Type 'exit', 'quit', or 'bye' to stop.\n")

question_count = 0

while True:
    try:
        question = input("You: ").strip()
    except (KeyboardInterrupt, EOFError):
        print("\n\n👋 Chat interrupted. Goodbye!")
        break

    # ── Exit conditions ────────────────────────────────────────────────────────
    if question.lower() in ["exit", "quit", "bye", "q"]:
        print(f"\n👋 Chat ended. You asked {question_count} question(s). Goodbye!")
        break

    # ── Validate input ─────────────────────────────────────────────────────────
    if not question:
        print("⚠️  Please type a question before pressing Enter.\n")
        continue

    # ── Generate RAG answer ────────────────────────────────────────────────────
    question_count += 1
    print()

    answer = ask_rag(question)

    print(f"🤖 Assistant: {answer}")
    print()

💬  RAG CHAT — Powered by Qwen via Groq
📄  Knowledge source : source.txt
🗄️  Vector store     : FAISS (33 chunks)
🤖  Model           : qwen/qwen3.6-27b

Ask questions about your document.
Type 'exit', 'quit', or 'bye' to stop.


📚 RETRIEVED CONTEXT
--------------------------------------------------------
[Chunk 1]
The Road Ahead

[Chunk 2]
India: A Comprehensive Overview

I. Introduction

[Chunk 3]
Digital Public Infrastructure:
In recent years, India has pioneered a unique model of Digital Public Infrastructure (DPI). The "India Stack"—comprising Aadhaar (biometric identity), UPI (Unified Payments Interface), and Data Empowerment architecture—has revolutionized financial inclusion, allowing millions to access banking and government services instantly.

VII. Challenges and Future Outlook

Despite its achievements, India faces significant hurdles.
--------------------------------------------------------

🤖 Assistant: <think>
Here's a thinking process:

1.  **Analyze User Input:**
   - Us

---
## 🏗️ Final Architecture Diagram

```
                  ┌─────────────────────┐
                  │   User Text File    │
                  │  (local disk path)  │
                  └──────────┬──────────┘
                             ↓
                  ┌─────────────────────┐
                  │   Text Extraction   │
                  │   + Cleaning        │
                  └──────────┬──────────┘
                             ↓
                  ┌─────────────────────┐
                  │   Text Chunking     │
                  │   chunk_size=500    │
                  │   overlap=50        │
                  └──────────┬──────────┘
                             ↓
                  ┌─────────────────────┐
                  │ Hugging Face        │
                  │ all-MiniLM-L6-v2   │
                  │ (384-dim vectors)   │
                  └──────────┬──────────┘
                             ↓
                  ┌─────────────────────┐
                  │    FAISS Vector DB  │
                  │  (in-memory index)  │
                  └──────────┬──────────┘
                             │
                  ┌──────────┴──────────┐
                  │     User Question   │
                  └──────────┬──────────┘
                             ↓
                  ┌─────────────────────┐
                  │ Query Embedding     │
                  │ (same MiniLM model) │
                  └──────────┬──────────┘
                             ↓
                  ┌─────────────────────┐
                  │ FAISS Similarity    │
                  │ Search (top-k=3)    │
                  └──────────┬──────────┘
                             ↓
                  ┌─────────────────────┐
                  │ Relevant Context    │
                  │ (top-3 chunks)      │
                  └──────────┬──────────┘
                             ↓
                  ┌─────────────────────┐
                  │ Augmented Prompt    │
                  │ System + Context    │
                  │ + Question          │
                  └──────────┬──────────┘
                             ↓
                  ┌─────────────────────┐
                  │ Qwen LLM via Groq   │
                  │ (qwen/qwen3-8b)     │
                  └──────────┬──────────┘
                             ↓
                  ┌─────────────────────┐
                  │ Final Answer        │
                  │ (grounded in doc)   │
                  └─────────────────────┘
```

---
## 📋 RAG Pipeline Summary

Here is the complete RAG pipeline you implemented:

```
INDEXING PHASE (runs once)
─────────────────────────────────────────────────────────
 1.  Load document            → .txt file from local path
 2.  Extract text             → Read from disk, clean whitespace
 3.  Split into chunks        → RecursiveCharacterTextSplitter
 4.  Generate embeddings      → all-MiniLM-L6-v2 (384-dim)
 5.  Store in FAISS           → In-memory vector index

RETRIEVAL + GENERATION PHASE (runs per question)
─────────────────────────────────────────────────────────
 6.  Accept user question     → Input from user
 7.  Embed the question       → Same MiniLM model
 8.  Search FAISS             → Top-3 most similar chunks
 9.  Retrieve context         → Concatenate top-k chunks
10.  Construct prompt         → System + Context + Question
11.  Send to Qwen via Groq    → Augmented prompt → LLM
12.  Generate answer          → Grounded in document evidence
```

---

## 🎓 Key Takeaways

| Concept | Lesson |
|---|---|
| **Why RAG?** | Gives LLMs access to private/current data without retraining |
| **Chunking** | Smaller focused chunks → better retrieval precision |
| **Embeddings** | Semantic vectors enable meaning-based (not keyword) search |
| **FAISS** | Efficient nearest-neighbour search over millions of vectors |
| **Prompt augmentation** | Context + question → grounded, accurate answers |
| **Hallucination prevention** | System instructions + retrieval grounding reduce invented facts |
| **Temperature** | Low temperature (0.2) keeps answers factual and focused |

---

## 🚀 Next Steps to Explore

- **Try different chunk sizes** — smaller chunks (200 chars) vs larger (1000 chars)
- **Increase TOP_K** — retrieve 5 or 7 chunks and observe answer quality
- **Add PDF support** — use `pypdf` or `pdfplumber` to extract text from PDFs
- **Persist the FAISS index** — save and reload the index with `vector_store.save_local()`
- **Add conversation memory** — pass chat history to maintain multi-turn context
- **Try reranking** — add a cross-encoder reranker to improve chunk selection quality
- **Metadata filtering** — tag chunks with page numbers or section headers
- **Use `.env` file** — manage your API key with `python-dotenv` for a cleaner local setup

---

*Built for educational demonstration of Retrieval-Augmented Generation (RAG).*  
*Stack: Qwen (Groq) · FAISS · all-MiniLM-L6-v2 · LangChain · Local Jupyter (VS Code)*